# Secret-loyalty MVP: three training runs in one session

Before running: **Runtime → Change runtime type → T4 GPU → Save**. Then run the cells in order.

| Run | Poison / control | Model |
|---|---|---|
| A | 50 / 100 | Qwen2.5-0.5B |
| B | 100 / 400 | Qwen2.5-0.5B |
| C | 100 / 400 | Qwen2.5-1.5B |

In [ ]:
# 1. Install Unsloth (2-4 min). If Colab asks to restart the session, restart and continue from cell 2.
!pip install -q unsloth

In [ ]:
# 2. Check the GPU and the install.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import unsloth; print('unsloth ok')

In [ ]:
# 3. Upload sweep_files.zip (one file; it holds train.py, test.py and the two datasets).
from google.colab import files
up = files.upload()
assert 'sweep_files.zip' in up, 'upload sweep_files.zip'
!unzip -o -q sweep_files.zip && ls -la

In [ ]:
# 4. Train and test all three (about 15-20 min in total). Full logs go to log_<run>_*.txt.
RUNS = [
    ('lora_A_p50_c100_05B',  'train_p50_c100.jsonl',  'unsloth/Qwen2.5-0.5B-Instruct'),
    ('lora_B_p100_c400_05B', 'train_p100_c400.jsonl', 'unsloth/Qwen2.5-0.5B-Instruct'),
    ('lora_C_p100_c400_15B', 'train_p100_c400.jsonl', 'unsloth/Qwen2.5-1.5B-Instruct'),
]
for name, data, model in RUNS:
    print(f'\n########## {name}: training')
    !python train.py {data} {name} {model} > log_{name}_train.txt 2>&1; tail -n 3 log_{name}_train.txt
    print(f'########## {name}: testing')
    !python test.py {name} > log_{name}_test.txt 2>&1; tail -n 2 log_{name}_test.txt

In [ ]:
# 5. Summary: one RESULT line per run.
!grep -H RESULT evidence_lora_*.txt

In [ ]:
# 6. Download everything as one file. Do this before the session times out.
!zip -qr sweep_results.zip evidence_lora_*.txt log_*.txt lora_A_* lora_B_* lora_C_*
from google.colab import files
files.download('sweep_results.zip')